# C-03 LangGraph

이 노트북은 LangGraph가 “LLM 호출 라이브러리”라기보다 “상태 기반 워크플로우를 만드는 도구”라는 점을 확인하는 예제입니다.

## LangGraph가 필요한 이유

메일 분석이 단순히 한 번의 LLM 호출로 끝나면 LangGraph가 필요하지 않을 수 있습니다. 하지만 실제 시스템에서는 분류, 중요도 판단, 담당 부서 추천, 사람 검토, 재시도, 저장 같은 단계가 생깁니다. LangGraph는 이런 단계를 노드와 엣지로 명시해 흐름을 관리합니다.

## 이 노트북에서 보는 포인트

- `EmailAnalysisState`라는 상태 객체를 정의합니다.
- 각 함수는 상태를 입력받아 일부 필드를 추가한 새 상태를 반환합니다.
- `StateGraph`에 노드를 등록하고 실행 순서를 엣지로 연결합니다.
- 마지막에 `app.invoke(sample)`로 전체 workflow를 실행합니다.

## 중요한 구분

이 예제는 LLM을 호출하지 않습니다. 일부러 규칙 기반 함수만 넣어 LangGraph의 구조를 먼저 보여 줍니다. 나중에는 `predict_email_intent` 같은 노드 내부에서 Ollama나 OpenAI 호출을 수행하도록 바꿀 수 있습니다.

## 언제 적합한가

- 처리 단계가 여러 개이고 상태를 이어 받아야 할 때
- 조건에 따라 사람 검토나 다른 분기로 보내야 할 때
- 재시도, 중단, 재개, 로그 추적이 필요한 agent workflow를 만들 때

단순한 단발성 분류 실험에는 과할 수 있지만, 운영 workflow가 복잡해질수록 장점이 커집니다.

In [ ]:
from typing import Literal, TypedDict

try:
    from langgraph.graph import END, StateGraph
except ImportError as exc:
    raise RuntimeError("이 노트북을 실행하려면 langgraph를 설치하세요") from exc

In [ ]:
class EmailAnalysisState(TypedDict, total=False):
    # 그래프의 모든 노드는 이 상태 딕셔너리를 입력받고 다시 반환합니다.
    # total=False라서 처음부터 모든 필드가 없어도 됩니다.
    subject: str
    body: str
    attachments: list[str]
    # predicted_email_intent는 임시 업무 분류 라벨입니다.
    # 후보값 예: inquiry, order, service, technical, other.
    # TODO: 실제 샘플 메일 검토 후 라벨명과 정의를 확정해야 합니다.
    predicted_email_intent: str
    # predicted_email_importance는 임시 우선순위 라벨입니다.
    # 후보값 예: low, normal, high, urgent.
    # TODO: 실제 SLA와 화면 정책에 맞게 기준을 확정해야 합니다.
    predicted_email_importance: str
    # predicted_assignee_area는 개인 담당자가 아니라 임시 담당 영역/팀 후보입니다.
    # TODO: 실제 라우팅 규칙 확정 후 assignee_user_id, assignee_team_id와 분리할지 결정해야 합니다.
    predicted_assignee_area: str
    # predicted_needs_human_review는 그래프가 계산한 사람 검토 필요성 예측값입니다.
    # 최종 상태 전이는 서비스 계층에서 별도 정책으로 확정해야 합니다.
    predicted_needs_human_review: bool


def predict_email_intent(state: EmailAnalysisState) -> EmailAnalysisState:
    # 실제 서비스에서는 이 위치에 LLM 분류 호출 또는 규칙+LLM 혼합 판단을 넣을 수 있습니다.
    # 아래 키워드와 predicted_email_intent 매핑은 LangGraph 구조를 보여 주기 위한 임시 예시입니다.
    # TODO: 사용자 검토를 거쳐 분류 라벨과 키워드/프롬프트 기준을 확정해야 합니다.
    text = f"{state['subject']} {state['body']}".lower()
    # 임시 정책: 누수/서비스/지원 키워드가 있으면 service, 아니면 inquiry로 둡니다.
    # 실제 운영에서는 order, technical, other까지 포함한 판단 기준이 필요합니다.
    predicted_email_intent = "service" if any(keyword in text for keyword in ["누수", "서비스", "지원"]) else "inquiry"
    return {**state, "predicted_email_intent": predicted_email_intent}


def predict_email_importance(state: EmailAnalysisState) -> EmailAnalysisState:
    text = f"{state['subject']} {state['body']}".lower()
    # 임시 정책: 긴급/즉시/급히 키워드가 있으면 urgent, 아니면 normal로 둡니다.
    # 실제 운영에서는 low/high/urgent의 SLA 기준과 예외 조건을 별도로 정해야 합니다.
    predicted_email_importance = "urgent" if any(keyword in text for keyword in ["긴급", "즉시", "급히"]) else "normal"
    return {**state, "predicted_email_importance": predicted_email_importance}


def suggest_assignee(state: EmailAnalysisState) -> EmailAnalysisState:
    # 앞 단계에서 만든 predicted_email_intent, predicted_email_importance를 사용해 `assignee_area`와 사람 검토 여부를 결정합니다.
    # 이 매핑도 임시 정책입니다. 실제 사용자/팀/업무 규칙이 정리되면 별도 라우팅 서비스로 분리해야 합니다.
    assignee = "service_team" if state.get("predicted_email_intent") == "service" else "sales_team"
    review = state.get("predicted_email_importance") == "urgent"
    return {**state, "predicted_assignee_area": assignee, "predicted_needs_human_review": review}

In [ ]:
# 그래프에 사용할 상태 타입을 지정합니다.
graph = StateGraph(EmailAnalysisState)

# 각 처리 단계를 노드로 등록합니다.
graph.add_node("predict_email_intent", predict_email_intent)
graph.add_node("predict_email_importance", predict_email_importance)
graph.add_node("suggest_assignee", suggest_assignee)

# 실행 순서를 엣지로 연결합니다.
graph.set_entry_point("predict_email_intent")
graph.add_edge("predict_email_intent", "predict_email_importance")
graph.add_edge("predict_email_importance", "suggest_assignee")
graph.add_edge("suggest_assignee", END)

# compile 후에는 일반 함수처럼 invoke할 수 있는 애플리케이션 객체가 됩니다.
app = graph.compile()

sample = {
    "subject": "밸브 누수 긴급 서비스 지원 요청",
    "body": "검사 중 밸브 누수가 발견되었습니다. 긴급 서비스 지원을 부탁드립니다.",
    "attachments": ["검사_보고서.pdf"],
}

result = app.invoke(sample)
result